# Data Cleaning - Capa BRONZE

En este notebook se realiza la limpieza inicial de los datos para la capa BRONZE.

El objetivo es preparar la información para su posterior uso en el pipeline ETL.

In [1]:
import os
# Configurar variables de entorno para Hadoop en Windows
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["hadoop.home.dir"] = "C:\\hadoop"
os.environ["PATH"] = f"{os.environ.get('PATH', '')};C:\\hadoop\\bin"

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, trim, lower, regexp_replace
from pyspark.ml.feature import Imputer
import os

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin")
    .getOrCreate()
)

print("Spark inicializado")

Spark inicializado


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/modified"
OUTPUT_PATH = "../data/processed/bronze"

# Crear directorio BRONZE si no existe
os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{INPUT_PATH}/users_data.csv")
)

transactions_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{INPUT_PATH}/transactions_data.csv")
)

cards_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{INPUT_PATH}/cards_data.csv")
)

print("Datos cargados")

Datos cargados


## Explorar la información

In [5]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|Female|       462 Rose Lane|   34.15|  -117.76|          $29,278|      $59,696|  $127,613|         787|               5|
|1746|         53|            68|      1966|         12|Female|3606 Federal Boul...|   40.76|   -73.74|          $37,891|      $77,254|  $191,349|         701|               5|
|1718|         81|            67|      1938|         11|Female|     766 Third Drive|   34.02|  -117.89|          $2

In [6]:
# Mostrar esquema

users_spark.printSchema()
cards_spark.printSchema()
transactions_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: string (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- total_debt: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: double (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: st

## Identificar valores faltantes

In [7]:
# Contar valores nulos en cada dataset

users_nulls = users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
])

cards_nulls = cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
])

transactions_nulls = transactions_spark.select([
    count(
        when(col(c).isNull(), c),
    ).alias(c)
    for c in transactions_spark.columns
])

print("Valores faltantes en users:")
users_nulls.show()

print("Valores faltantes en cards:")
cards_nulls.show()

print("Valores faltantes en transactions:")
transactions_nulls.show()

Valores faltantes en users:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|         63|            65|        63|         55|    61|     68|      50|       54|               64|           46|        63|          49|              62|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+

Valores faltantes en cards:
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+--------------

## Limpieza de datos para capa BRONZE

In [8]:
# Limpiar strings: quitar espacios y convertir a minúsculas
def clean_string(df, column):
    return df.withColumn(column, trim(lower(col(column))))

# Limpiar moneda: quitar $ y comas
def clean_currency(df, column):
    return df.withColumn(column, regexp_replace(regexp_replace(col(column), "\\$", ""), ",", "").cast("float"))

# Imputar con moda
def impute_mode(df, column):
    mode_value = (
        df.filter(col(column).isNotNull())
        .groupBy(column)
        .count()
        .orderBy(col("count").desc())
        .first()[column]
    )
    return df.fillna({column: mode_value})

In [9]:
# Limpiar users_spark

# Limpiar strings
users_spark = clean_string(users_spark, "gender")
users_spark = clean_string(users_spark, "address")

# Limpiar columnas de moneda
currency_columns = ["per_capita_income", "yearly_income", "total_debt"]
for column in currency_columns:
    if column in users_spark.columns:
        users_spark = clean_currency(users_spark, column)

# Imputar current_age con mediana
imputer = Imputer(
    inputCols=["current_age"],
    outputCols=["current_age"]
).setStrategy("median")

users_spark = imputer.fit(users_spark).transform(users_spark)

# Imputar gender con moda
users_spark = impute_mode(users_spark, "gender")

# Validar rangos
users_spark = users_spark.withColumn(
    "current_age",
    when(col("current_age") < 18, 18)
    .when(col("current_age") > 120, 120)
    .otherwise(col("current_age"))
)

users_spark = users_spark.withColumn(
    "credit_score",
    when(col("credit_score") < 300, 300)
    .when(col("credit_score") > 850, 850)
    .otherwise(col("credit_score"))
)

print("Limpieza de users completada")

Limpieza de users completada


In [10]:
# Limpiar cards_spark

# Limpiar strings
string_columns = ["card_brand", "card_type", "has_chip", "card_on_dark_web"]
for column in string_columns:
    if column in cards_spark.columns:
        cards_spark = clean_string(cards_spark, column)

# Limpiar credit_limit
if "credit_limit" in cards_spark.columns:
    cards_spark = clean_currency(cards_spark, "credit_limit")

# Imputar num_cards_issued con mediana
imputer = Imputer(
    inputCols=["num_cards_issued"],
    outputCols=["num_cards_issued"]
).setStrategy("median")

cards_spark = imputer.fit(cards_spark).transform(cards_spark)

# Imputar card_brand con moda
cards_spark = impute_mode(cards_spark, "card_brand")

# Validar credit_limit
cards_spark = cards_spark.withColumn(
    "credit_limit",
    when(col("credit_limit") < 0, 0)
    .otherwise(col("credit_limit"))
)

print("Limpieza de cards completada")

Limpieza de cards completada


In [11]:
# Limpiar transactions_spark

# Limpiar strings
string_columns = ["use_chip", "merchant_city", "merchant_state", "zip", "errors"]
for column in string_columns:
    if column in transactions_spark.columns:
        transactions_spark = clean_string(transactions_spark, column)

# Limpiar amount
if "amount" in transactions_spark.columns:
    transactions_spark = transactions_spark.withColumn(
        "amount",
        regexp_replace(regexp_replace(col("amount"), "\\$", ""), ",", "").cast("float")
    )

# Imputar amount con mediana
imputer = Imputer(
    inputCols=["amount"],
    outputCols=["amount"]
).setStrategy("median")

transactions_spark = imputer.fit(transactions_spark).transform(transactions_spark)

# Imputar merchant_city con moda
transactions_spark = impute_mode(transactions_spark, "merchant_city")

# Validar amount
transactions_spark = transactions_spark.withColumn(
    "amount",
    when(col("amount") < -10000, -10000)
    .when(col("amount") > 10000, 10000)
    .otherwise(col("amount"))
)

print("Limpieza de transactions completada")

Limpieza de transactions completada


## Verificar los cambios

In [12]:
# Verificar valores nulos despues de la limpieza

users_nulls_clean = users_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_spark.columns
])

cards_nulls_clean = cards_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_spark.columns
])

transactions_nulls_clean = transactions_spark.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_spark.columns
])

print("Valores faltantes en users despues de limpieza:")
users_nulls_clean.show()

print("Valores faltantes en cards despues de limpieza:")
cards_nulls_clean.show()

print("Valores faltantes en transactions despues de limpieza:")
transactions_nulls_clean.show()

Valores faltantes en users despues de limpieza:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|          0|            65|        63|         55|     0|     68|      50|       54|               64|           46|        63|          49|              62|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+

Valores faltantes en cards despues de limpieza:
+---+---------+----------+---------+-----------+-------+---+--------+----------------+--

In [13]:
# Mostrar muestra de datos limpios

print("Muestra de users limpios:")
users_spark.show(5)

print("Muestra de cards limpios:")
cards_spark.show(5)

print("Muestra de transactions limpios:")
transactions_spark.show(5)

Muestra de users limpios:
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|female|       462 rose lane|   34.15|  -117.76|          29278.0|      59696.0|  127613.0|         787|               5|
|1746|         53|            68|      1966|         12|female|3606 federal boul...|   40.76|   -73.74|          37891.0|      77254.0|  191349.0|         701|               5|
|1718|         81|            67|      1938|         11|female|     766 third drive|   34

## Guardar datos en capa BRONZE

In [14]:
# Guardar datos limpios en formato Parquet

users_spark.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/users.parquet")
cards_spark.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/cards.parquet")
transactions_spark.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/transactions.parquet")

print("Datos guardados en capa BRONZE")

Datos guardados en capa BRONZE


## Finalizar sesión de Spark

In [15]:
# Finalizar Spark
spark.stop()
print("Pipeline completado")

Pipeline completado
